# DB15K clean — AdaMF KGC và truy hồi ảnh với CLIP

Chọn **Colab GPU**, kết nối Drive và chạy các ô theo thứ tự hoặc **Run all**. Dữ liệu nguồn chỉ được đọc; kết quả lưu trên Drive tại đường dẫn `RUN` in ở ô cấu hình.

Mô hình học từ các triple của clean train bằng loss KGC, sau đó cung cấp điểm quan hệ để xếp hạng lại ảnh ứng viên của CLIP. Bản này dùng **AdaMF không MAT**, không huấn luyện bằng retrieval cross-entropy.

Query gián tiếp dùng anchor, relation và hướng dự đoán để chấm entity của ảnh ứng viên. Query trực tiếp giữ thứ hạng CLIP. Không đưa entity đáp án vào scorer.

Các khác biệt với GitHub gốc: clean split, sampler Python Bernoulli lọc triple clean train, features đóng băng, visual features được gộp từ CLIP ảnh train và mask modality thiếu. Text features canonical có giới hạn về bằng chứng thứ tự hàng theo handoff. Đây là tích hợp AdaMF vào retrieval, không phải tái lập nguyên cấu hình AdaMF-MAT.

## 1. Kết nối Drive và cấu hình

Ô dưới cài thư viện, kiểm tra GPU và thiết lập đường dẫn. Giữ nguyên `RUN` để tiếp tục checkpoint cũ. Chỉ tăng giới hạn huấn luyện khi muốn tiếp tục lâu hơn; thay model, dữ liệu hoặc cấu hình học khác cần run riêng.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b1_adamf_db15k_seed2027', 1, globals())


## 2. Kiểm tra dữ liệu clean và các file tham chiếu

Đọc `training_interface.json`, manifest retrieval và canonical references; kiểm tra hash trước khi dùng dữ liệu. Nạp mapping entity/relation, kiểm tra clean train và validation không trùng triple.

Ảnh liên kết với entity thông qua bảng `entity_image`. CLIP image features được dùng chung qua tham chiếu; textual features được đọc theo kích thước thực tế. Visual đầu vào mô hình chỉ gộp từ ảnh train để tránh đưa ảnh validation/test vào quá trình học lớp chiếu.

Các dòng query/relevance/candidate thuộc test được loại khỏi phần phát triển. Notebook không đánh giá official test. Metadata CLIP ảnh chưa khóa commit chính xác; hạn chế này được ghi lại khi tạo cache text encoder.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b1_adamf_db15k_seed2027', 3, globals())


## 3. Tạo cache CLIP cho validation

Mã hóa query validation bằng CLIP đóng băng, khóa revision text encoder và lưu cache. Tính baseline trên toàn bộ candidate pool validation, đồng thời lưu top-k để rerank.

Cache có identity gắn với dữ liệu, encoder và cấu hình top-k. Các mô hình so sánh cần dùng cùng query IDs, candidate pool, CLIP features/cache và protocol.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b1_adamf_db15k_seed2027', 5, globals())


## 4. Mô hình AdaMF và loss KGC

Scorer `AdvMixRotatE` và `SigmoidLoss` được nhúng từ [mã nguồn AdaMF-MAT](https://github.com/zjukg/AdaMF-MAT). Lớp điều chỉnh thêm mask modality thiếu, đóng băng features và tính biểu diễn theo entity ID duy nhất để giảm bộ nhớ.

Sampler thay head hoặc tail theo xác suất Bernoulli và loại các triple đã có trong clean train. Không đọc test để tạo hoặc lọc negative. Smoke check kiểm tra forward/backward trước khi huấn luyện.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b1_adamf_db15k_seed2027', 7, globals())


## 5. Đánh giá retrieval và chọn trọng số kết hợp

Với query head/tail, AdaMF chấm triple chứa entity của từng ảnh ứng viên. Điểm CLIP và AdaMF được chuẩn hóa z-score theo query trước khi kết hợp:

`điểm cuối = alpha × điểm CLIP + (1 − alpha) × điểm AdaMF`.

Query trực tiếp giữ nguyên baseline CLIP. Nếu top-k không chứa ảnh relevant, giữ best-positive rank của baseline. Ties dùng optimistic rank: một cộng số điểm lớn hơn điểm positive tốt nhất.

Báo cáo MRR và Recall@K theo combined, direct, indirect, head, tail, seen-only và cold-start. Recall@K ở đây là query hit-rate theo best-positive rank. Target entity chỉ dùng tạo nhãn và chia nhóm đánh giá, không làm đầu vào chấm điểm.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b1_adamf_db15k_seed2027', 9, globals())


## 6. Huấn luyện, checkpoint và tự động resume

Ô dưới chọn checkpoint có tiến độ cao nhất còn khôi phục được từ các bản last, previous, pending và thư mục checkpoint mốc. Checkpoint chứa model, optimizer, lịch sử, trạng thái lấy mẫu và vị trí batch.

Hệ thống lưu định kỳ trong quá trình train, trước/sau validation và tại các mốc dự phòng. Các mốc cũ được giữ lại. Best model và alpha được chọn bằng combined validation retrieval MRR; không chọn bằng test.

Nếu phiên bị ngắt, chạy lại notebook với cùng `RUN`. Nếu hết thời lượng phiên, notebook lưu tiến độ rồi tạm dừng; chạy lại để tiếp tục. Nếu tất cả checkpoint đều mất hoặc hỏng, chương trình dừng thay vì âm thầm train lại từ đầu. Không chạy đồng thời hai phiên ghi vào cùng run.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b1_adamf_db15k_seed2027', 11, globals())


## 7. Xem kết quả validation

Ô cuối xuất bảng CLIP và CLIP + AdaMF ở checkpoint tốt nhất, cùng biểu đồ loss KGC và validation retrieval MRR. Có thể xem kết quả sau khi phiên huấn luyện tạm dừng nếu đã có validation hoàn tất.

Các file chính trong `RUN`: `best_adamf.pt`, `last_checkpoint.pt`, `previous_checkpoint.pt`, `epoch_checkpoints/`, `history.json`, `best_validation_metrics.csv` và `training_validation.png`.

MRR ở đây đo image retrieval, không so trực tiếp với KGC MRR của paper. Query trực tiếp dùng CLIP nên phần đóng góp AdaMF cần được xem riêng trên nhóm gián tiếp. Kết quả một seed chưa thay thế thực nghiệm nhiều seed; bộ query đánh giá thủ công, ablation, đánh giá test và xAI là các bước tiếp theo.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b1_adamf_db15k_seed2027', 13, globals())


## 8. Đánh giá cuối cùng trên tập test

Ô code bên dưới nạp checkpoint AdaMF tốt nhất và giữ nguyên trọng số kết hợp α đã chọn trên validation để so sánh **CLIP** với **CLIP + AdaMF** trên tập test.

Không huấn luyện lại hoặc điều chỉnh siêu tham số trên test. Kết quả gồm **MRR, Recall@1, Recall@5, Recall@10 và Recall@50**, được báo cáo chung và tách theo nhóm truy vấn trực tiếp, gián tiếp, head và tail.

Các bảng kết quả, thứ hạng từng truy vấn và cấu hình đánh giá được lưu trong thư mục `test_locked_<timestamp>/` bên trong `RUN`.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b1_adamf_db15k_seed2027', 15, globals())
